# P10｜PAFA Projector + Loss

**Pipeline ID：P10**  
**研究问题：** 在 P9 固定 projector 架构上，仅加入 PAFA patient-aware PCSL/GPAL loss，能否在具备合法 patient/group ID 的 sources 上改善 native task 且不破坏其他 lanes？  
**Role：** loss one-axis comparator。  
**状态：Design / Not Ready。**

## 合同、组件与唯一变化

**Verified Contract：** PAFA source defaults 使用 patient-aware PCSL/GPAL（当前 pipeline spec 记录 `0.1*PCSL + 0.1*GPAL`）；patient identity 缺失时不得用 date/device/file proxy 伪造。

**Proposed Method：** 复制 P9 的 BEATs encoder、projector、native heads、sampler、预算、seed 与 selection；唯一变化为在合法 patient-ID rows 上启用冻结权重的 PCSL/GPAL。HF 没有 patient ID，date proxy 只能 grouping，因此 HF 的 PAFA loss 必须 masked off；native loss仍保留。Comparator 为 **P10 − P9，仅 patient-aware loss 改变**。

**HOLD：** exact PAFA code revision、loss normalization、eligible datasets/rows、batch patient composition、weight、trainable scope、go/no-go 待冻结；不得用 outer result 调 weight。

| Dataset | Unit / head | Patient-loss eligibility |
|---|---|---|
| ICBHI | cycle / `[B,4]` | patient ID 可用；官方 split overlap 披露 |
| SPRSound | event / `[B,2]` + `[B,7]` | patient/group ID receipt 通过后候选 |
| HF | 15-s recording / observed-positive heads | **不 eligible**；date proxy 不是 patient；gap omitted |
| KAUH | recording / raw9 `[B,9]` | P-number；B/D/E 同组；shared/diagnosis HOLD |

**Execution gates：** P9 freeze、PAFA source/SHA、patient-ID eligibility matrix、loss batch invariants、matched compute、local smoke、independent verifier 与执行授权全部通过；否则 fail closed。

### P9/P10 matched projector scope lock

P10 必须逐 lane、逐 row 复用 P9 的 `real_patient_id_eligible_lanes_only` projector mask 与 `projector_scope_receipt` hash；没有真实 patient ID 的 lanes 在 P9/P10 都保持完全相同的 P2 native bypass。P10−P9 的唯一变化是在同一 eligible rows 上增加冻结的 PCSL/GPAL；projector eligibility、eligible/bypass row counts、unique-patient counts、projector/native-only call counts 必须逐 lane 相等。HF 在两者中都不进入 projector，也不进入 patient-aware loss。任何 scope/hash/count 不一致都使 P10−P9 attribution fail closed。

In [ ]:
import os
from pathlib import Path
PIPELINE_ID = "P10"
NOTEBOOK = Path("reproduce/P10_pafa_projector_plus_loss.ipynb")
STATUS = "Design / Not Ready"
PROJECT_ROOT = Path.cwd() if Path.cwd().name != "reproduce" else Path.cwd().parent
DATASET_ROOT = Path(os.environ.get("ACOUSTIC_DATA_ROOT", "dataset/raw"))
CONFIG = Path("experiments/P10_pafa_projector_plus_loss.yaml")
APPROVAL = Path("result/approvals/P10_execution_authorization.json")
EXECUTION_ALLOWED = False
dry_run_plan = {"pipeline_id": PIPELINE_ID, "comparison": "P10 minus P9; PCSL/GPAL only", "source_default_weights_to_freeze": {"PCSL":0.1,"GPAL":0.1}, "hf_patient_loss_allowed": False, "missing_gates": [n for n,p in {"config":PROJECT_ROOT/CONFIG,"approval":PROJECT_ROOT/APPROVAL}.items() if not p.is_file()]}
assert NOTEBOOK.name.startswith(f"{PIPELINE_ID}_") and not EXECUTION_ALLOWED
dry_run_plan


## Outputs / receipt schema / claim boundary

未来 receipt：P9 upstream hash、PAFA code/checkpoint provenance、patient-ID source、eligible/masked row counts、PCSL/GPAL denominators/weights、matched compute、native metrics、verification 与 decision。

**Claim boundary：** 仅是合法 patient-ID surface 上 PAFA losses 相对 projector-only 的效果；不支持 HF patient-aware claim、universal generalization、official PAFA reproduction 或 zero-shot。

**Test Result=Not run**  
**Decision：Not evaluated；Design / Not Ready。**